# Experiment G: Expand generic labels to L/R using trained RF

1. Train 10-class RF on 535 L/R-labeled cycles
2. Predict L/R for 1571 generic-labeled cycles
3. Retrain on all 2106 cycles with L/R labels
4. Compare: 535 only vs 2106 expanded

In [ ]:
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
print('Setup done')

In [ ]:
import glob, re, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

In [ ]:
# ── Functions ─────────────────────────────────────────────────
CONFIG = {
    'FS': 50.0, 'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_SAVGOL_WINDOW': 21, 'CYCLE_SAVGOL_POLYORDER': 3,
    'CYCLE_MIN_PEAK_DEGS': 300.0, 'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0, 'TARGET_LEN': 64, 'MIN_CYCLE_SAMPLES': 10,
}
KNOWN_PATTERNS = {'overhand','sneak_overhand','underhand','sneak_underhand','dragon_roll','underhand_default'}
LR_CLASSES = {'UR','UL','OR','OL','USR','USL','OSR','OSL','FB','BF'}
GENERIC_TO_COARSE = {'U_generic':'underhand','O_generic':'overhand','DR_generic':'dragon_roll',
                     'SU_generic':'sneak_underhand','SO_generic':'sneak_overhand'}
COARSE_MAP = {'UR':'underhand','UL':'underhand','OR':'overhand','OL':'overhand',
    'USR':'sneak_underhand','USL':'sneak_underhand','OSR':'sneak_overhand','OSL':'sneak_overhand',
    'FB':'dragon_roll','BF':'dragon_roll'}
# Which L/R labels are valid for each generic label
GENERIC_TO_LR = {
    'U_generic': ['UR', 'UL'], 'O_generic': ['OR', 'OL'],
    'DR_generic': ['FB', 'BF'], 'SU_generic': ['USR', 'USL'], 'SO_generic': ['OSR', 'OSL'],
}
FINE_MAP = {'UR':'UR','ur':'UR','UR0':'UR','UR-CW':'UR','UL0':'UL',
    'OR':'OR','or':'OR','OR2':'OR','OR3':'OR','OR-OSL':'OR','OR/OSL?':'OR',
    'OL':'OL','ol':'OL','OL2':'OL','USR':'USR','USL':'USL','OSR':'OSR','OSL':'OSL',
    'FB':'FB','fb':'FB','FB2':'FB','BF2':'BF','bf':'BF',
    'underhand':'U_generic','overhand':'O_generic','dragon_roll':'DR_generic',
    'sneak_underhand':'SU_generic','sneak_overhand':'SO_generic',
    'CW':None,'cw':None,'CCW':None,'ccw':None,'idle':None,'Idle3':None,'Idle-or-ol?':None,'VQ5':None,'vq16':None}
_FP = [(re.compile(r'^USR$',re.I),'USR'),(re.compile(r'^USL$',re.I),'USL'),
    (re.compile(r'^OSR$',re.I),'OSR'),(re.compile(r'^OSL$',re.I),'OSL'),
    (re.compile(r'^UR',re.I),'UR'),(re.compile(r'^UL',re.I),'UL'),
    (re.compile(r'^OR',re.I),'OR'),(re.compile(r'^OL',re.I),'OL'),
    (re.compile(r'^FB',re.I),'FB'),(re.compile(r'^BF',re.I),'BF'),
    (re.compile(r'^CW$',re.I),None),(re.compile(r'^CCW$',re.I),None),
    (re.compile(r'^idle',re.I),None),(re.compile(r'^vq',re.I),None)]
def map_fine(raw):
    raw = raw.strip()
    if raw in FINE_MAP: return FINE_MAP[raw]
    if raw.lower() in FINE_MAP: return FINE_MAP[raw.lower()]
    for p,c in _FP:
        if p.match(raw): return c
    return None
def extract_signals(df):
    t=df['timestamp_ms'].values/1000.0
    A=df[['ax_w','ay_w','az_w']].values
    om=df[['gx','gy','gz']].values*(np.pi/180.0)
    return t,A,om
def detect_cycles(t,om,fs=50.0):
    md=np.linalg.norm(om,axis=1)*(180/np.pi); n=len(md)
    if n<7: return []
    w=int(CONFIG['CYCLE_SAVGOL_WINDOW'])
    if w%2==0: w+=1
    w=max(5,min(w,n if n%2==1 else n-1))
    po=max(1,min(int(CONFIG['CYCLE_SAVGOL_POLYORDER']),w-2))
    ms=savgol_filter(md,window_length=w,polyorder=po,mode='interp')
    ms=savgol_filter(ms,window_length=w,polyorder=po,mode='interp')
    pks,_=find_peaks(ms,distance=max(1,int(CONFIG['CYCLE_MIN_PERIOD_S']*fs)),
        prominence=CONFIG['CYCLE_PROMINENCE_DEGS'],height=CONFIG['CYCLE_MIN_PEAK_DEGS'])
    if len(pks)==0: return []
    cyc=[]
    for i,p in enumerate(pks):
        l=0 if i==0 else (pks[i-1]+p)//2
        r=(len(t)-1) if i==len(pks)-1 else (p+pks[i+1])//2
        if r>l and (r-l)>=CONFIG['MIN_CYCLE_SAMPLES']: cyc.append((l,r))
    return cyc
def pair_cycles(t0,c0,t1,c1):
    p0,p1,u=[],[],set()
    for a in c0:
        bi,bo=-1,-1.0
        for i,b in enumerate(c1):
            if i in u: continue
            o=max(0,min(t0[a[1]],t1[b[1]])-max(t0[a[0]],t1[b[0]]))
            if o>bo: bo,bi=o,i
        if bi>=0 and bo>0: p0.append(a);p1.append(c1[bi]);u.add(bi)
    return p0,p1
def resample(s,n):
    if len(s)<2: return np.zeros(n) if s.ndim==1 else np.zeros((n,s.shape[1]))
    xo,xn=np.linspace(0,1,len(s)),np.linspace(0,1,n)
    if s.ndim==1: return np.interp(xn,xo,s)
    return np.column_stack([np.interp(xn,xo,s[:,j]) for j in range(s.shape[1])])
def build_cm(A0,A1,om0,om1,s0,e0,s1,e1):
    tl=CONFIG['TARGET_LEN']
    r0=resample(np.column_stack([A0[s0:e0],om0[s0:e0]]),tl)
    r1=resample(np.column_stack([A1[s1:e1],om1[s1:e1]]),tl)
    return np.column_stack([r0,r1]).T.astype(np.float32)
print('Functions defined')

In [ ]:
# ── Load all cycles with fine labels ──────────────────────────
all_matrices=[]; all_fine=[]; all_groups=[]
def load_c(d0p,d1p,name,fine='unlabeled',windows=None):
    df0,df1=pd.read_csv(d0p),pd.read_csv(d1p)
    t0,A0,om0=extract_signals(df0); t1,A1,om1=extract_signals(df1)
    c0=detect_cycles(t0,om0,CONFIG['FS']); c1=detect_cycles(t1,om1,CONFIG['FS'])
    p0,p1=pair_cycles(t0,c0,t1,c1)
    if windows:
        fp0,fp1=[],[]
        for (s0,e0),(s1,e1) in zip(p0,p1):
            if any(ws<=(t0[s0]+t0[e0])/2<we for ws,we in windows): fp0.append((s0,e0));fp1.append((s1,e1))
        p0,p1=fp0,fp1
    g=name.split('/')[0] if '/' in name else name; cnt=0
    for (s0,e0),(s1,e1) in zip(p0,p1):
        all_matrices.append(build_cm(A0,A1,om0,om1,s0,e0,s1,e1))
        all_fine.append(fine); all_groups.append(g); cnt+=1
    return cnt
# Homogeneous
for d0 in sorted(glob.glob(os.path.join(DATA_PROCESSED,'*_device0_processed.csv'))):
    d1=d0.replace('_device0_','_device1_')
    if not os.path.exists(d1): continue
    stem=os.path.basename(d0).replace('_device0_processed.csv','')
    parts=stem.split('_')
    if len(parts)<3: continue
    rest='_'.join(parts[2:]); fine='unlabeled'
    for pat in sorted(KNOWN_PATTERNS,key=len,reverse=True):
        if rest.startswith(pat):
            if pat in('underhand','underhand_default'): fine='U_generic'
            elif pat=='overhand': fine='O_generic'
            elif pat=='dragon_roll': fine='DR_generic'
            elif pat=='sneak_underhand': fine='SU_generic'
            elif pat=='sneak_overhand': fine='SO_generic'
            break
    if fine=='unlabeled': continue
    load_c(d0,d1,stem,fine)
# Heterogeneous
if os.path.isdir(NEW_LABELED_RAW):
    for sn in sorted(os.listdir(NEW_LABELED_RAW)):
        sd=os.path.join(NEW_LABELED_RAW,sn)
        if not os.path.isdir(sd): continue
        lp=None
        for fn in('labels_corrected.json','labels.json'):
            p=os.path.join(sd,fn)
            if os.path.isfile(p): lp=p; break
        if not lp: continue
        d0=os.path.join(DATA_PROCESSED,sn+'_device0_processed.csv')
        d1=os.path.join(DATA_PROCESSED,sn+'_device1_processed.csv')
        if not(os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lp,encoding='utf-8') as f: data=_json.load(f)
        segs=data.get('segments',data) if isinstance(data,dict) else data
        wbl=defaultdict(list)
        for seg in segs:
            fi=map_fine(seg.get('label',''))
            if fi is None: continue
            s,e=seg.get('start'),seg.get('end')
            if s is None: continue
            if e is None: e=s+2.0
            wbl[fi].append((float(s),float(e)))
        for fi,wins in sorted(wbl.items()): load_c(d0,d1,sn+'/'+fi,fi,wins)

X_all = np.array([cm.ravel() for cm in all_matrices])
y_fine = np.array(all_fine)
g_all = np.array(all_groups)

lr_mask = np.array([l in LR_CLASSES for l in y_fine])
generic_mask = np.array([l in GENERIC_TO_COARSE for l in y_fine])

print(f'Total labeled: {len(X_all)}')
print(f'L/R labeled: {lr_mask.sum()}')
print(f'Generic labeled: {generic_mask.sum()}')
print(f'\nL/R distribution:')
for l,c in sorted(Counter(y_fine[lr_mask]).items(), key=lambda x:-x[1]):
    print(f'  {l:<6s}: {c:>4d}')
print(f'\nGeneric distribution:')
for l,c in sorted(Counter(y_fine[generic_mask]).items(), key=lambda x:-x[1]):
    print(f'  {l:<12s}: {c:>4d}')

In [ ]:
# ── Step 1: Train 10-class RF on L/R-labeled data ────────────

X_lr = X_all[lr_mask]; y_lr = y_fine[lr_mask]; g_lr = g_all[lr_mask]
sc = StandardScaler(); X_lr_s = sc.fit_transform(X_lr)

rf = RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42)
rf.fit(X_lr_s, y_lr)
print(f'Trained 10-class RF on {len(X_lr)} L/R cycles')
print(f'Classes: {list(rf.classes_)}')

In [ ]:
# ── Step 2: Predict L/R for generic-labeled cycles ───────────

X_gen = X_all[generic_mask]; y_gen = y_fine[generic_mask]; g_gen = g_all[generic_mask]
X_gen_s = sc.transform(X_gen)

pred_lr = rf.predict(X_gen_s)
pred_proba = rf.predict_proba(X_gen_s)
pred_conf = np.max(pred_proba, axis=1)

# Constrain: predicted L/R must be consistent with generic label
# e.g., U_generic can only become UR or UL, not OR
y_expanded = []
conf_expanded = []
valid_expanded = []
for i in range(len(X_gen)):
    gen_label = y_gen[i]
    allowed = set(GENERIC_TO_LR.get(gen_label, []))
    pred = pred_lr[i]
    conf = pred_conf[i]
    if pred in allowed:
        y_expanded.append(pred)
        conf_expanded.append(conf)
        valid_expanded.append(True)
    else:
        # Pick the highest-probability allowed class instead
        class_probs = {cls: pred_proba[i, list(rf.classes_).index(cls)] 
                       for cls in allowed if cls in rf.classes_}
        if class_probs:
            best = max(class_probs, key=class_probs.get)
            y_expanded.append(best)
            conf_expanded.append(class_probs[best])
            valid_expanded.append(True)
        else:
            y_expanded.append('unknown')
            conf_expanded.append(0)
            valid_expanded.append(False)

y_expanded = np.array(y_expanded)
conf_expanded = np.array(conf_expanded)
valid_expanded = np.array(valid_expanded)

print(f'Generic cycles: {len(X_gen)}')
print(f'Valid expansions: {valid_expanded.sum()}')
print(f'Mean confidence: {conf_expanded[valid_expanded].mean():.3f}')
print(f'\nExpanded label distribution:')
for l,c in sorted(Counter(y_expanded[valid_expanded]).items(), key=lambda x:-x[1]):
    print(f'  {l:<6s}: {c:>4d} (mean conf={conf_expanded[(y_expanded==l)&valid_expanded].mean():.3f})')

# Confidence histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(conf_expanded[valid_expanded], bins=30, color='#7F77DD', alpha=0.7)
ax.axvline(0.7, color='red', ls='--', label='0.7 threshold')
ax.axvline(0.9, color='orange', ls='--', label='0.9 threshold')
ax.set_xlabel('Prediction confidence'); ax.set_ylabel('Count')
ax.set_title('RF confidence on generic cycles'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Step 3: Build expanded datasets at different confidence thresholds ─

THRESHOLDS = [0.0, 0.5, 0.7, 0.8, 0.9]

for thresh in THRESHOLDS:
    keep = valid_expanded & (conf_expanded >= thresh)
    n_added = keep.sum()
    print(f'Threshold {thresh:.1f}: {n_added} generic cycles added (total: {lr_mask.sum() + n_added})')

In [ ]:
# ── Step 4: LOSO evaluation at each threshold ────────────────

def run_loso_expanded(X_lr, y_lr, g_lr, X_gen, y_exp, g_gen, conf, thresh, n_classes=10):
    """LOSO on L/R data, with expanded generic data added to training only."""
    keep_gen = conf >= thresh
    unique_groups = np.unique(g_lr)
    class_groups = defaultdict(set)
    for l,g in zip(y_lr,g_lr): class_groups[l].add(g)
    singleton = {c for c,gs in class_groups.items() if len(gs)<2}
    test_groups = [g for g in unique_groups if not set(y_lr[g_lr==g]).issubset(singleton)]
    
    at, ap = [], []
    at5, ap5 = [], []  # coarse 5-class
    for g in test_groups:
        te = g_lr == g; tr = ~te
        # Training: L/R labeled (not test group) + expanded generic (not test group)
        gen_tr = keep_gen & (g_gen != g)
        X_tr = np.vstack([X_lr[tr], X_gen[gen_tr]])
        y_tr = np.concatenate([y_lr[tr], y_exp[gen_tr]])
        X_te = X_lr[te]; y_te = y_lr[te]
        
        sc2 = StandardScaler(); X_tr_s = sc2.fit_transform(X_tr); X_te_s = sc2.transform(X_te)
        clf = RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42)
        clf.fit(X_tr_s, y_tr)
        preds = clf.predict(X_te_s)
        at.extend(y_te.tolist()); ap.extend(preds.tolist())
        # Map to 5-class
        at5.extend([COARSE_MAP[l] for l in y_te])
        ap5.extend([COARSE_MAP.get(p, p) for p in preds])
    
    at,ap = np.array(at),np.array(ap)
    at5,ap5 = np.array(at5),np.array(ap5)
    labs10 = sorted(set(at)|set(ap)); labs5 = sorted(set(at5)|set(ap5))
    f1_10 = f1_score(at, ap, average='macro', labels=labs10, zero_division=0)
    f1_5 = f1_score(at5, ap5, average='macro', labels=labs5, zero_division=0)
    acc_10 = np.mean(at==ap); acc_5 = np.mean(at5==ap5)
    n_train = lr_mask.sum() - len(y_lr[g_lr==test_groups[0]]) + keep_gen.sum()  # approximate
    return {'f1_10': f1_10, 'f1_5': f1_5, 'acc_10': acc_10, 'acc_5': acc_5,
            'cm_10': confusion_matrix(at,ap,labels=labs10), 'labs_10': labs10,
            'cm_5': confusion_matrix(at5,ap5,labels=labs5), 'labs_5': labs5,
            'report_10': classification_report(at,ap,labels=labs10,zero_division=0),
            'report_5': classification_report(at5,ap5,labels=labs5,zero_division=0),
            'n_added': keep_gen.sum()}

results = {}
print('Running LOSO at each confidence threshold...')
print(f'{"Thresh":>6s}  {"Added":>6s}  {"Total":>6s}  {"F1-10":>6s}  {"F1-5":>6s}  {"Acc-10":>6s}  {"Acc-5":>6s}')
for thresh in THRESHOLDS:
    keep = valid_expanded & (conf_expanded >= thresh)
    r = run_loso_expanded(X_lr, y_lr, g_lr, X_gen[valid_expanded], y_expanded[valid_expanded],
                          g_gen[valid_expanded], conf_expanded[valid_expanded], thresh)
    results[thresh] = r
    print(f'{thresh:>6.1f}  {r["n_added"]:>6d}  {lr_mask.sum()+r["n_added"]:>6d}  '
          f'{r["f1_10"]:>6.3f}  {r["f1_5"]:>6.3f}  {r["acc_10"]:>6.3f}  {r["acc_5"]:>6.3f}')

In [ ]:
# ── Plots ─────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
thrs = list(results.keys())
f1_10s = [results[t]['f1_10'] for t in thrs]
f1_5s = [results[t]['f1_5'] for t in thrs]
ax1.plot(thrs, f1_10s, 'o-', color='#7F77DD', label='10-class F1', lw=2)
ax1.plot(thrs, f1_5s, 's-', color='#5DCAA5', label='5-class F1', lw=2)
ax1.axhline(0.700, color='#7F77DD', ls='--', alpha=0.5, label='E1 10-class (0.700)')
ax1.axhline(0.811, color='#5DCAA5', ls='--', alpha=0.5, label='E1 5-class (0.811)')
ax1.set_xlabel('Confidence threshold'); ax1.set_ylabel('LOSO F1 macro')
ax1.set_title('F1 vs confidence threshold'); ax1.legend(fontsize=8)

n_added = [results[t]['n_added'] for t in thrs]
ax2.bar(range(len(thrs)), n_added, color='#E8A838')
ax2.set_xticks(range(len(thrs))); ax2.set_xticklabels([str(t) for t in thrs])
ax2.set_xlabel('Confidence threshold'); ax2.set_ylabel('Generic cycles added')
ax2.set_title('Training data expansion')
plt.tight_layout(); plt.show()

# Best threshold confusion matrix
best_t = max(results, key=lambda t: results[t]['f1_5'])
r = results[best_t]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for ax, cm, labs, title in [
    (ax1, r['cm_10'], r['labs_10'], f'10-class (thresh={best_t}, F1={r["f1_10"]:.3f})'),
    (ax2, r['cm_5'], r['labs_5'], f'5-class (thresh={best_t}, F1={r["f1_5"]:.3f})')]:
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs)))
    ax.set_xticklabels(labs, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(labs, fontsize=8)
    for i in range(len(labs)):
        for j in range(len(labs)):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',
                    color='white' if cm[i,j]>cm.max()*0.5 else 'black',fontsize=7)
    ax.set_title(title); plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# ── SUMMARY ──────────────────────────────────────────────────
print('='*70)
print('EXPERIMENT G: EXPAND GENERIC LABELS SUMMARY')
print('='*70)
print(f'L/R labeled (ground truth): {lr_mask.sum()}')
print(f'Generic labeled (to expand): {generic_mask.sum()}')
print(f'Valid expansions: {valid_expanded.sum()}')
print(f'Mean confidence on generic: {conf_expanded[valid_expanded].mean():.3f}')
print(f'')
print(f'{"Thresh":>6s}  {"Added":>6s}  {"Total":>6s}  {"F1-10":>6s}  {"F1-5":>6s}')
print('-'*40)
for t in THRESHOLDS:
    r = results[t]
    print(f'{t:>6.1f}  {r["n_added"]:>6d}  {lr_mask.sum()+r["n_added"]:>6d}  {r["f1_10"]:>6.3f}  {r["f1_5"]:>6.3f}')
print(f'')
best_t_10 = max(results, key=lambda t: results[t]['f1_10'])
best_t_5 = max(results, key=lambda t: results[t]['f1_5'])
print(f'Best 10-class: thresh={best_t_10}, F1={results[best_t_10]["f1_10"]:.3f} (E1 baseline: 0.700)')
print(f'Best 5-class:  thresh={best_t_5}, F1={results[best_t_5]["f1_5"]:.3f} (E1 baseline: 0.811)')
print(f'')
d10 = results[best_t_10]['f1_10'] - 0.700
d5 = results[best_t_5]['f1_5'] - 0.811
print(f'Delta 10-class vs E1: {d10:+.3f} ({"BETTER" if d10>0 else "WORSE" if d10<0 else "SAME"})')
print(f'Delta 5-class vs E1:  {d5:+.3f} ({"BETTER" if d5>0 else "WORSE" if d5<0 else "SAME"})')
print(f'V08 reference: F1=0.632')